# 競馬AI 学習ノートブック
**セルを上から順番に実行してください（▶ボタンを押す）**

- セル1〜2: 準備（1分）
- セル3: データ収集（数時間 ※放置でOK）
- セル4: 血統データ取得（1〜2時間）
- セル5: モデル学習（30分〜2時間）
- セル6: artifact共有方針の確認（GitHubへmodelをpushしない）


In [ ]:
# セル1: ライブラリインストール＆Googleドライブ接続
!pip install -q requests beautifulsoup4 pandas numpy lightgbm scikit-learn joblib

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/keiba-ai/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/keiba-ai/models', exist_ok=True)
print('✅ 準備完了')

In [ ]:
# セル2: GitHubからコードを取得
!git clone https://github.com/yamaguchikei88-web/keiba-ai.git /content/keiba-ai 2>/dev/null || \
  (cd /content/keiba-ai && git pull)

import sys
sys.path.insert(0, '/content/keiba-ai/scraper')
sys.path.insert(0, '/content/keiba-ai/ml')
print('✅ コード取得完了')

In [ ]:
# セル3: データ収集（10年分）
# ※ 途中で止まっても再実行すれば続きから始まります
from pathlib import Path
import netkeiba_scraper as scraper

scraper.DB_PATH = Path('/content/drive/MyDrive/keiba-ai/data/keiba.db')
scraper.init_db()

# 収集する年を指定（最初は2020年〜で試してもOK）
START_YEAR = 2015
END_YEAR = 2025

print(f'データ収集開始: {START_YEAR}〜{END_YEAR}年')
print('Googleドライブに保存されるので途中でセッションが切れても大丈夫です')
scraper.scrape_all(start_year=START_YEAR, end_year=END_YEAR)
print('✅ データ収集完了')

In [ ]:
# セル4: 血統データ補完
from pathlib import Path
import horse_scraper as hs

hs.DB_PATH = Path('/content/drive/MyDrive/keiba-ai/data/keiba.db')
print('血統データ取得開始...')
hs.fill_pedigree()
print('✅ 血統データ完了')

In [ ]:
# セル5: AIモデル学習
import sys
from pathlib import Path

# パスをGoogleドライブに向ける
import features
features.DB_PATH = Path('/content/drive/MyDrive/keiba-ai/data/keiba.db')

import train as trainer
trainer.DB_PATH = Path('/content/drive/MyDrive/keiba-ai/data/keiba.db')
trainer.MODEL_DIR = Path('/content/drive/MyDrive/keiba-ai/models')
trainer.MODEL_PATH = trainer.MODEL_DIR / 'keiba_lgbm.pkl'
trainer.STATS_CACHE_PATH = trainer.MODEL_DIR / 'stats_cache.pkl'
trainer.META_PATH = trainer.MODEL_DIR / 'model_meta.json'
trainer.MODEL_DIR.mkdir(exist_ok=True)

print('AI学習開始（数万回の繰り返し学習）...')
model, auc = trainer.train()
print(f'✅ 学習完了！ 精度スコア(AUC): {auc:.4f}')
print('モデルはGoogleドライブに保存されました')

In [ ]:
# セル6: artifact共有方針の確認
# 学習済みmodel/cache/metadataをGitHubへコピー・pushしない。
# 承認済みの共有Google Drive / Cloud Storageに保持し、
# artifact hash・model version・実験結果だけをregistryへ記録する。
from pathlib import Path
model_dir = Path('/content/drive/MyDrive/keiba-ai/models')
print('共有artifact保存先:', model_dir)
print('GitHubにはコード・文書・registry migrationのみをcommitします。')